# RNN and Natural Language Processing

In this exercise we will try to classify IMDB dataset: Given the text of a review, can you predict if the review was positive or negative?

Before doing this exercise, you might want to become more familier with LSTMs by considering the example FlightPassengerPredictions.

The data for this exercise can be found here:
https://sid.erda.dk/share_redirect/encok5nw3y

***

Last Update: 2026-11-11

In [5]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import pickle
from collections import defaultdict
from torch import nn
import json
from torch.utils.data import DataLoader
from tqdm import tqdm
from itertools import chain
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"

## Data

In [6]:
limit_data = 10000  # limit the amount of data for speed, change as you please

def remove_special_symbols(string):
    return ''.join(s for s in string if ord(s)>96 and ord(s)<123 or s == ' ')

with open('train.json') as f:
    train_text, train_labels = json.load(f)
    idxs = np.random.permutation(len(train_text))
    train_text = [remove_special_symbols(train_text[i]) for i in idxs[:limit_data]]
    train_labels = [train_labels[i] for i in idxs[:limit_data]]
    
with open('test.json') as f:
    test_text, test_labels = json.load(f)
    idxs = np.random.permutation(len(test_text))   
    test_text = [remove_special_symbols(test_text[i]) for i in idxs[:limit_data]]
    test_labels = [test_labels[i] for i in idxs[:limit_data]]

Let's have a look at the data... Here is a negative review (label = 0)

In [7]:
print(train_text[0])
print(train_labels[0])

ne of the previous reviewers wrote that there appeared to be no middle ground for opinions of ove tory one loved it or hated it ut there seems to be a remarkable distribution of opinions throughout the scale of  to  or me this movie rated a  here are some beautiful scenes and locations and ay illand turns in a fabulous job as livers father ut the movie did not do a particularly compelling job of telling its story and the story was not so unique as to warrant multiple viewings at least not for me  may be a bit of a snob but  tend to avoid movies with yan eal   still havent seen arry yndon  because most of them but not all are ruined for me by his presence he lone exception is hats p oc in which his straight performance is the perfect underlining for arbra treisands goofball protagonist  and not coincidentally he takes a shot at ove tory for good measure craw and eal tend to mug their lines rather than act thembr br his movie is notable for the beginning of one fine career it was ommy ee

And a positive one:

In [8]:
print(train_text[3])
print(train_labels[3])

fter having seen arrying afia d nearly lost my faith in orean movie business ut this one brought my faith backbr br eading female character who is university student forced to teach the spoiled rich but charistmatic high school student guy e is actually female characters age hrough some hilarious quarrells these two end up being great friendsbr br  was pleasantly surprised that newcomer kwan sang woo did his job with excellency e was such a revelation ctress im ha nul was also charming as usualbr br his movie tried hard to avoid any cliches that can be seen in typical romantic comedies nd it didnt show us any unnecessary nude sex scenes t was brilliant lovely  fresh made me want to see it againbr br 
1


## Embedding

Let's have a look at all the words we have:

In [9]:
all_words = list(chain(*[x.lower().split() for x in train_text]))
print('total number of unique words =', len(set(all_words)))

total number of unique words = 74760


That's a lot of words... of course, we could clean the data even more if we wanted to. But we won't...
(for instance, there are probably many misspelled words)

In [10]:
Counter(all_words)

Counter({'the': 115635,
         'a': 62247,
         'and': 61803,
         'of': 57119,
         'to': 53124,
         'is': 42628,
         'in': 34829,
         'he': 27573,
         'that': 27077,
         'it': 25593,
         'this': 24001,
         'br': 23541,
         'was': 18987,
         'as': 17199,
         'with': 17176,
         'his': 17099,
         'for': 16676,
         'movie': 16445,
         'film': 14688,
         'but': 13485,
         'on': 13302,
         'are': 11699,
         'not': 10987,
         'have': 10976,
         'you': 10636,
         'be': 10564,
         'one': 9572,
         'an': 9012,
         'at': 8838,
         'by': 8629,
         'all': 8366,
         'from': 7892,
         'who': 7834,
         'like': 7581,
         'or': 7559,
         'its': 7245,
         'her': 7110,
         'they': 7107,
         'so': 6792,
         'about': 6643,
         'just': 6611,
         'has': 6508,
         'out': 6371,
         'some': 5797,
        

In [11]:
n_words = 25000   # let's make a model that only understand 25000 words

In [12]:
words, count = np.unique(all_words, return_counts=True)
idxs = np.argsort(count)[-n_words:]
vocab = ['<UNK>'] + list(words[idxs][::-1])
print(vocab[:5], '...', vocab[-5:])

['<UNK>', 'the', 'a', 'and', 'of'] ... ['evangelical', 'nobleman', 'willy', 'lague', 'ntz']


Not very surprisingly, the most commonly used word is _the_. The 25000th most used word is _chimneys_. We have added a special word `<UNK>` which we will used to mark words outside our vocabulary.

We can now turn a sentence into a sequence of integers that correspond to the position in the vocab.

In [13]:
vocab_d = {vocab[i]: i for i in range(len(vocab))}  # for quick look-up
def sentence_to_integer_sequence(s):
    return torch.tensor([vocab_d[x] if x in vocab_d else 0 for x in s.split()], dtype=torch.long)

In [14]:
sentence_to_integer_sequence("i really liked the movie xenopus51")

tensor([146,  56, 409,   1,  18,   0])

We are now representing words in a "25000"-dimensional space: we have a unique integer for each word we can represent. To reduce this complexity, we instead intend to represent each word as a 50-dimensional real vector. Pytorch to the rescue:

In [15]:
embedding = nn.Embedding(len(vocab), 50)

`nn.Embedding` assigns are random, trainable vector to each word. For instance:

In [16]:
print(embedding(sentence_to_integer_sequence("movie")))

tensor([[-0.3729, -1.7484,  0.3394,  0.5183,  0.2129,  1.1231,  0.4480, -0.2156,
          0.5472, -0.3252, -1.2791,  1.2848, -1.3101,  0.7475, -0.4190, -0.4432,
          1.9113,  0.4540, -0.3773,  0.4058,  0.4266, -0.0547,  1.0890, -0.8417,
          0.2074,  1.0695, -0.0927,  1.2101, -0.2451, -0.6121,  0.1019,  0.2849,
          0.0974,  0.9015,  2.0183,  0.3031, -0.0622,  1.0296,  0.5245, -0.3936,
          0.1529,  1.6402, -2.0034,  0.0649,  0.5000,  0.1376, -0.2756,  1.9260,
         -0.7459,  1.2100]], grad_fn=<EmbeddingBackward0>)


The special `<UNK>` word, which signifies unknown we can choose to zero out:

In [17]:
embedding.weight.data[0, :] = 0

Here is an example of how we represent a sentence then:

In [18]:
print(embedding(sentence_to_integer_sequence("i really liked the movie xenopus51")).shape)
print(embedding(sentence_to_integer_sequence("i really liked the movie xenopus51")))

torch.Size([6, 50])
tensor([[-0.3333,  1.3739,  0.1731, -2.7524, -0.9879, -1.0427, -0.0582,  0.7657,
          0.3803, -0.5848, -0.2534,  0.2474, -0.0473, -1.9548, -1.2905,  0.5161,
          0.6248, -0.3473,  0.9451, -0.2764,  1.0142,  0.2177, -0.2547,  0.1686,
          1.1343,  0.5223,  0.0912, -1.6447,  0.4927,  1.2771, -0.4034, -0.8036,
         -0.4130, -0.0407,  0.4653,  0.8077,  1.9931,  0.0432,  0.5354, -0.1086,
          0.0529,  0.0599,  0.4355, -0.6767,  0.3539,  0.0884, -0.9664, -0.1929,
         -2.2424,  0.3205],
        [-0.0782,  1.4711,  0.1097, -0.6215, -0.5772, -1.9562, -0.0650,  1.1038,
          0.0864, -0.6136, -1.2289, -0.8463,  0.2418, -1.4966, -2.4817, -2.2797,
         -0.0719,  1.1296, -2.5394, -1.7315,  1.2726, -0.0209,  1.5922, -0.2573,
          1.3200, -0.6918,  0.1362,  0.4793,  1.1613,  0.7198, -1.0375,  0.3051,
         -0.8669, -1.5877, -0.2525,  0.9504,  0.2999, -0.2960,  0.8330,  2.0129,
         -0.5330, -0.6786, -0.1164,  0.6993, -0.5974,  0.9554

In this way, the sentence bascially becomes a 6x50 pixel image.

You can now use `nn.LSTM` after an embedding to define a neural network for sentenes.

This network will have a _lot_ of parameters. For each word, 50 parameters needs to be trained, and then comes the LSTM on top of that.
This is the reason that _transfer learning_ is so important in natural language processing (NLP).

Perhaps the simplest form of transfer learning is to use a pretrained embedding layer.

In [19]:
with open('glove.6B.50d.pkl', 'rb') as f:
    glove = pickle.load(f)

In [20]:
glove['movie']

array([ 0.30824 ,  0.17223 , -0.23339 ,  0.023105,  0.28522 ,  0.23076 ,
       -0.41048 , -1.0035  , -0.2072  ,  1.4327  , -0.80684 ,  0.68954 ,
       -0.43648 ,  1.1069  ,  1.6107  , -0.31966 ,  0.47744 ,  0.79395 ,
       -0.84374 ,  0.064509,  0.90251 ,  0.78609 ,  0.29699 ,  0.76057 ,
        0.433   , -1.5032  , -1.6423  ,  0.30256 ,  0.30771 , -0.87057 ,
        2.4782  , -0.025852,  0.5013  , -0.38593 , -0.15633 ,  0.45522 ,
        0.04901 , -0.42599 , -0.86402 , -1.3076  , -0.29576 ,  1.209   ,
       -0.3127  , -0.72462 , -0.80801 ,  0.082667,  0.26738 , -0.98177 ,
       -0.32147 ,  0.99823 ])

These pretrained word embeddings will have a good structure to them. For instance:

In [21]:
print('Distance b/w queen and prince =', np.linalg.norm(glove['queen'] - glove['prince']))
print('Distance b/w movie and prince =', np.linalg.norm(glove['movie'] - glove['prince']))

Distance b/w queen and prince = 3.3926491063677284
Distance b/w movie and prince = 6.450784821820618


You can even sometimes get away with doing algebra with these vectors:

In [22]:
queenlike = glove['king'] - glove['man'] + glove['woman']

In [23]:
print('Distance b/w queen and algebraic queen =', np.linalg.norm(glove['queen'] - queenlike))
print('Distance b/w queen and king =', np.linalg.norm(glove['queen'] - glove['king']))

Distance b/w queen and algebraic queen = 2.8391206432941996
Distance b/w queen and king = 3.4777562289742345


We can fill out our embedding layer using these pretrained vectors:

In [24]:
filled = 0
not_found = []
for i, w in enumerate(vocab):
    if w in glove:
        embedding.weight.data[i, :] = torch.tensor(glove[w], dtype=torch.float)
        filled += 1
    else:
        not_found.append(w)
print(f'Of the words in the vocab, {100 * filled / (len(vocab) - 1)} % were updated using Glove vectors')
print('Examples of words not found :', not_found[1:7])

Of the words in the vocab, 73.636 % were updated using Glove vectors
Examples of words not found : ['owever', 'ichael', 'itbr', 'ichard', 'ritish', 'nfortunately']


Clearly the words we do not find are due to misspellings. You now have three choices before you continue with the exercise:

(1) Use _hunspell_ or similar to fix misspelled words

(2) Change the vocabulary (`vocab`) to be based on words that are both frequent in the text and have glove vectors

(3) Ignore the issue.

Finally, with Glove vectors we do not need to train the embedding. We can consider it fixed:

In [25]:
embedding.requires_grad_(False)

Embedding(25001, 50)

### Exercise 1

Train a neural network using `nn.LSTM` layers to classify the IMDB reviews.

Ideas:

 - The sentences can be quite long, so you might want to limit them to, say, maximum 100 words
 - `nn.LSTM` can take batched input, but be careful with `batch_first=True/False`.
 - If batched input are used, they normally have to be equal in size. You can put sentences to always be 100 words long, by adding `<UNK>` words to short sentences.
 - Alternatively, `nn.LSTM` can also accept batched, variable-length input using `torch.nn.utils.rnn.pack_sequence`.
 - (a finaly, albeit slow, alternative is to simple run the LSTM on un-batched input)


In [26]:
words, count = np.unique(all_words, return_counts=True)
words = words[np.argsort(count)[::-1]]
vocab = ['<UNK>']
for w in words:
    if w in glove:
        vocab.append(w)    
    if len(vocab) == n_words:
        break

print(vocab[:10])
print(vocab[-10:])

['<UNK>', 'the', 'a', 'and', 'of', 'to', 'is', 'in', 'he', 'that']
['strokes', 'trafficker', 'wittiest', 'pacific', 'nastiest', 'wittier', 'seriocomic', 'plucks', 'wireless', 'transcended']


In [27]:
embedding = nn.Embedding(len(vocab), 50)
embedding.requires_grad_(False)
for i, w in enumerate(vocab):
    if w in glove:
        embedding.weight[i, :] = torch.tensor(glove[w], dtype=embedding.weight.dtype)
embedding.weight[0, :] = 0

In [28]:
vocab_d = {vocab[i]: i for i in range(len(vocab))}  # for quick look-up
max_seq_len = 100

def sentence_to_integer_sequence(s):
    return torch.tensor([vocab_d[x] if x in vocab_d else 0 for x in s.split()][:max_seq_len], dtype=torch.long)


train_seq = [sentence_to_integer_sequence(x) for x in train_text]
test_seq = [sentence_to_integer_sequence(x) for x in test_text]

In [29]:
class IMDBNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(50, 30, num_layers=2, batch_first=True, bidirectional=True)
        self.linear = nn.Linear(2 * 30, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = embedding(x)
        output, _ = self.lstm(x)
        final_hidden = output[:, -1, :]
        return self.sigmoid(self.linear(final_hidden)[:, 0])
    
net = IMDBNet()
embedding.to(device)
net.to(device)

IMDBNet(
  (lstm): LSTM(50, 30, num_layers=2, batch_first=True, bidirectional=True)
  (linear): Linear(in_features=60, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [30]:
from torch.nn.utils.rnn import pad_sequence
train = pad_sequence(train_seq, batch_first=True)
test = pad_sequence(test_seq, batch_first=True)

In [31]:
from torch.utils.data import DataLoader
train_loader = DataLoader(list(zip(train, train_labels)), batch_size=128)
test_loader = DataLoader(list(zip(test, test_labels)), batch_size=128)

In [32]:
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

loss_criterion = nn.BCELoss()
net.to(device)
for epoch in range(100):
    acc_loss = 0.0
    for seq, label in tqdm(train_loader):
        seq = seq.to(device)
        label = label.to(device).float()
        
        output = net(seq)
        loss = loss_criterion(output, label)
        loss.backward()  # calculate gradients: d Loss / d Paramters
        opt.step()  # take a step down-hill
        opt.zero_grad()  # zero the gradient calculations for next iteration
        acc_loss += float(loss)
    acc_loss /= len(train)
    print(f'Epoch {epoch + 1} loss = {acc_loss}')

100%|██████████████████████████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 84.29it/s]


Epoch 1 loss = 0.005417141026258469


100%|█████████████████████████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 193.36it/s]


Epoch 2 loss = 0.00501084856390953


100%|█████████████████████████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 215.79it/s]


Epoch 3 loss = 0.005115352499485016


... (190 outputs left out)

100%|█████████████████████████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 187.15it/s]


Epoch 99 loss = 0.002287066824734211


100%|█████████████████████████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 142.70it/s]

Epoch 100 loss = 0.002090504012256861


In [33]:
acc = 0
count = 0
for seq, label in tqdm(train_loader):
    seq = seq.to(device)
    label = label.to(device)
    output = net(seq)
    acc += torch.sum(torch.eq(torch.round(output), label))
    count += len(label)
acc = acc / count
print('Train accuracy =', acc)

acc = 0
count = 0
for seq, label in tqdm(test_loader):
    seq = seq.to(device)
    label = label.to(device)
    output = net(seq)
    acc += torch.sum(torch.eq(torch.round(output), label))
    count += len(label)
acc = acc / count
print('Test accuracy =', acc)

100%|█████████████████████████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 286.62it/s]


Train accuracy = tensor(0.8991, device='cuda:0')


100%|█████████████████████████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 424.06it/s]


Test accuracy = tensor(0.7462, device='cuda:0')


In [34]:
embedding.cpu()
net.cpu()

IMDBNet(
  (lstm): LSTM(50, 30, num_layers=2, batch_first=True, bidirectional=True)
  (linear): Linear(in_features=60, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [35]:
float(net(sentence_to_integer_sequence("fargo is a great comedy")[None, :]))

0.9996985197067261

In [36]:
float(net(sentence_to_integer_sequence("worst movie i have ever seen")[None, :]))

0.5832049250602722

In [37]:
float(net(sentence_to_integer_sequence("best movie i have ever seen")[None, :]))

0.9688566327095032

In [38]:
float(net(sentence_to_integer_sequence("i don't really know how i feel about this movie")[None, :]))

0.9993228912353516

In [39]:
float(net(sentence_to_integer_sequence("at first i thought this would be terrible, but then it turned out to be really nice")[None, :]))

0.4477343261241913

In [40]:
float(net(sentence_to_integer_sequence("at first i thought this would be good, but then it turned out to be really terrible")[None, :]))

0.07698296010494232

## Why did that go so wrong?

The scores above are pretty striking — "worst movie I have ever seen" comes out positive, similar to "best movie I have ever seen". The model has clearly not learned sentiment. Here's what happened.

### The padding problem: reading zeros instead of words

After `pad_sequence`, every sequence in the batch is padded with zeros out to the length of the longest sequence. "Worst movie I have ever seen" is only 6 words; in a batch where the longest review is 100 words, it looks like this to the model:

```
worst  movie  i  have  ever  seen  <PAD>  <PAD>  ...  <PAD>
  ^                                                       ^
position 0                                          position 99
```

Our `forward` method then does:

```python
final_hidden = output[:, -1, :]   # <-- position 99, pure padding
```

By the time the LSTM reaches position 99, it has been reading zeros for 93 steps. Whatever signal it accumulated from "worst" at position 0 has been heavily diluted. The model is classifying sentiment based almost entirely on the padding tail, not the actual words.

### The model learns frequency, not meaning

Because the last hidden state is so dominated by padding, the model falls back on whatever correlates with the padding pattern — essentially review length and the frequency of whichever words happened to appear in non-padding positions. Long, elaborate reviews are more likely to be positive (enthusiastic fans write more). Common words like "movie", "ever", "seen" appear in both positive and negative reviews, so the model can't distinguish them. Almost everything ends up scoring high.

### The negation problem: apostrophes disappear

There's a second, quieter issue. `remove_special_symbols` keeps only lowercase a–z and spaces, so `"don't"` becomes `"dont"`, which almost certainly isn't in the GloVe vocabulary and maps to `<UNK>` → zeroed out. Negations silently vanish. "I don't really know" becomes "I really know", which the model might actually read as mildly positive.

### The fix: mean pooling over real positions

Instead of taking the hidden state at the last (padded) position, we average the LSTM output over all *real* (non-padding) positions. Every word contributes equally regardless of where it falls in the sequence, and padding contributes nothing. This is the single change that matters most — everything else below is secondary.


In [41]:
# --- Fixed model: mean pooling over non-padding positions ---
# The forward signature now takes `lengths` so it knows which positions are real.

class IMDBNetFixed(nn.Module):
    def __init__(self, hidden_size=128):
        super().__init__()
        self.lstm    = nn.LSTM(50, hidden_size, num_layers=2,
                               batch_first=True, bidirectional=True, dropout=0.3)
        self.dropout = nn.Dropout(0.4)
        self.linear  = nn.Linear(2 * hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, lengths):
        x = embedding(x)                                    # (B, T, 50)
        output, _ = self.lstm(x)                            # (B, T, 2*hidden)

        # Build a mask that is True for real positions, False for padding
        T    = output.size(1)
        mask = torch.arange(T, device=x.device)[None, :] < lengths[:, None]
        mask = mask.unsqueeze(-1).float()                   # (B, T, 1)

        # Average over real positions only
        pooled = (output * mask).sum(dim=1) / mask.sum(dim=1)   # (B, 2*hidden)
        return self.sigmoid(self.linear(self.dropout(pooled))[:, 0])

net_fixed = IMDBNetFixed(hidden_size=128)
embedding.to(device)
net_fixed.to(device)
print(net_fixed)


IMDBNetFixed(
  (lstm): LSTM(50, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (linear): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [42]:
# We need to pass sequence lengths through the DataLoader so the model
# knows where padding begins. A custom collate_fn handles this cleanly.

from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded  = pad_sequence(seqs, batch_first=True)
    return padded, torch.tensor(labels, dtype=torch.float), lengths

train_loader_fixed = DataLoader(list(zip(train_seq, train_labels)),
                                batch_size=128, collate_fn=collate_fn, shuffle=True)
test_loader_fixed  = DataLoader(list(zip(test_seq,  test_labels)),
                                batch_size=128, collate_fn=collate_fn)


In [43]:
opt_fixed      = torch.optim.Adam(net_fixed.parameters(), lr=3e-4)
scheduler      = torch.optim.lr_scheduler.ReduceLROnPlateau(
                     opt_fixed, patience=3, factor=0.5, verbose=True)
loss_criterion = nn.BCELoss()
net_fixed.to(device)

for epoch in range(30):
    net_fixed.train()
    acc_loss = 0.0
    for seq, label, lengths in tqdm(train_loader_fixed, leave=False):
        seq, label, lengths = seq.to(device), label.to(device), lengths.to(device)
        output = net_fixed(seq, lengths)
        loss   = loss_criterion(output, label)
        loss.backward()
        opt_fixed.step()
        opt_fixed.zero_grad()
        acc_loss += float(loss)
    avg_loss = acc_loss / len(train_loader_fixed)

    # Validation loss for the scheduler
    net_fixed.eval()
    val_loss = 0.0
    with torch.no_grad():
        for seq, label, lengths in test_loader_fixed:
            seq, label, lengths = seq.to(device), label.to(device), lengths.to(device)
            val_loss += float(loss_criterion(net_fixed(seq, lengths), label))
    val_loss /= len(test_loader_fixed)

    scheduler.step(val_loss)
    print(f'Epoch {epoch+1:2d} | train loss {avg_loss:.4f} | val loss {val_loss:.4f}')


C:\Users\cls8dx\AppData\Local\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch  1 | train loss 0.6611 | val loss 0.6143


Epoch  2 | train loss 0.5791 | val loss 0.5790


Epoch  3 | train loss 0.5473 | val loss 0.5629


Epoch  4 | train loss 0.5302 | val loss 0.5518


Epoch  5 | train loss 0.5249 | val loss 0.5392


... (21 outputs left out)

Epoch 27 | train loss 0.3877 | val loss 0.4710


Epoch 28 | train loss 0.3796 | val loss 0.4702


Epoch 29 | train loss 0.3734 | val loss 0.4717


Epoch 30 | train loss 0.3743 | val loss 0.4999


In [44]:
net_fixed.eval()
embedding.cpu()
net_fixed.cpu()

for split, loader in [('Train', train_loader_fixed), ('Test', test_loader_fixed)]:
    acc, count = 0, 0
    with torch.no_grad():
        for seq, label, lengths in loader:
            output = net_fixed(seq, lengths)
            acc   += torch.sum(torch.eq(torch.round(output), label)).item()
            count += len(label)
    print(f'{split} accuracy = {acc/count:.4f}')


Train accuracy = 0.8179
Test accuracy = 0.7618


In [45]:
# Same sentences as before — now let's see if the model has actually learned something.

def score(sentence):
    net_fixed.eval()
    with torch.no_grad():
        seq    = sentence_to_integer_sequence(sentence)[None, :]
        length = torch.tensor([seq.size(1)])
        return float(net_fixed(seq, length))

test_sentences = [
    "fargo is a great comedy",
    "worst movie i have ever seen",
    "best movie i have ever seen",
    "i dont really know how i feel about this movie",
    "at first i thought this would be terrible but then it turned out to be really nice",
    "at first i thought this would be good but then it turned out to be really terrible",
]

for s in test_sentences:
    print(f'{score(s):.4f}  "{s}"')


0.9598  "fargo is a great comedy"
0.0022  "worst movie i have ever seen"
0.9937  "best movie i have ever seen"
0.1318  "i dont really know how i feel about this movie"
0.0109  "at first i thought this would be terrible but then it turned out to be really nice"
0.0072  "at first i thought this would be good but then it turned out to be really terrible"


## Language model

Often in NLP, a lot of unlabelled text is available. We can use this to pretrain the model, before fine-tuning to the task at hand.

In [46]:
with open('unlabelled.json') as f:
    text = json.load(f)
print(len(text))

50000


A simple way to pretrain is to train a _language model_. This is model that tries to predict the next word in a sentence. For instance, given:

In [47]:
" ".join(text[2].split()[:28])

'This is still a pretty bad and silly simplistic typical slasher but when being compared to the previous sequel "Slumber Party Massacre II" this movie is a step'

can you guess the next word?

In a language model, we consider the above the input, and the out is:

In [48]:
text[2].split()[28]

'up'

The input we encode using the embedding, while the output is a probability map over words.
In other words, the last layer is something like `nn.Linear(..., len(vocab))`.

### Exercise 2 (optional)

Train a language model.

After you have trained the model, try to make to complete sentences.

## Transfer learning
### Exercise 3 (optional)

Discard the last layer of the now trained language model and use it to train on the original IMDB-problem.